In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CSV_PATH = "master_US.csv"

df = pd.read_csv(CSV_PATH)
df["DATE"] = pd.to_datetime(df["DATE"])
df = df.sort_values("DATE").reset_index(drop=True)

# Keep only the columns we need for the first stage
data = df[["DATE", "CPI", "Customs_Duties", "Imports_Goods"]].copy()

print(data.head())

print(data.info())

        DATE    CPI  Customs_Duties  Imports_Goods
0 1960-01-01  29.37           1.176         15.700
1 1960-02-01  29.41             NaN            NaN
2 1960-03-01  29.41             NaN            NaN
3 1960-04-01  29.54           1.084         15.888
4 1960-05-01  29.57             NaN            NaN
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 792 entries, 0 to 791
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   DATE            792 non-null    datetime64[ns]
 1   CPI             791 non-null    float64       
 2   Customs_Duties  264 non-null    float64       
 3   Imports_Goods   264 non-null    float64       
dtypes: datetime64[ns](1), float64(3)
memory usage: 24.9 KB
None


In [3]:
# create tariff rate raw number, only quarterly though
data["tariff_rate_raw"] = data["Customs_Duties"] / data["Imports_Goods"]

#filter data just for after 2020
current_data = data.where(data["DATE"] >= "1999-01-01")

#some command so that we dont fill with all nas
current_data = current_data.dropna(subset=["DATE"]).reset_index(drop=True)

# fill in blanks using linear method
current_data["tariff_rate"] = current_data["tariff_rate_raw"].interpolate(method = "linear")

print(current_data.head())

        DATE    CPI  Customs_Duties  Imports_Goods  tariff_rate_raw  \
0 1999-01-01  164.7          18.216        978.939         0.018608   
1 1999-02-01  164.7             NaN            NaN              NaN   
2 1999-03-01  164.8             NaN            NaN              NaN   
3 1999-04-01  165.9          17.856       1024.016         0.017437   
4 1999-05-01  166.0             NaN            NaN              NaN   

   tariff_rate  
0     0.018608  
1     0.018218  
2     0.017827  
3     0.017437  
4     0.017723  


In [4]:
forecast_horizon = 3
current_data["cpi_future"] = current_data["CPI"].shift(-forecast_horizon)

#create shifted tariff rates - so in month a, the row has the current tariff rate / 
#tariff rate 3 months before, 6 months before, and 12 months before /
# this enables that we can use old tariff rates to predict current CPIs
current_data["tau_lag_3"] = current_data["tariff_rate"].shift(3)
current_data["tau_lag_6"] = current_data["tariff_rate"].shift(6)
current_data["tau_lag_12"] = current_data["tariff_rate"].shift(12)

# repeat with CPI so we can use old price levels to predict current price levels
current_data["cpi_lag_3"] = current_data["CPI"].shift(1)
current_data["cpi_lag_6"] = current_data["CPI"].shift(3)
current_data["cpi_lag_12"] = current_data["CPI"].shift(12)

#filter data just for after 2020
century_data = current_data.where(current_data["DATE"] >= "2000-01-01")
century_data = century_data.dropna(subset=["DATE"]).reset_index(drop=True)

print(century_data.head())

        DATE    CPI  Customs_Duties  Imports_Goods  tariff_rate_raw  \
0 2000-01-01  169.3          19.952       1193.644         0.016715   
1 2000-02-01  170.0             NaN            NaN              NaN   
2 2000-03-01  171.0             NaN            NaN              NaN   
3 2000-04-01  170.9          21.960       1232.293         0.017820   
4 2000-05-01  171.2             NaN            NaN              NaN   

   tariff_rate  cpi_future  tau_lag_3  tau_lag_6  tau_lag_12  cpi_lag_3  \
0     0.016715       170.9   0.018627   0.018294    0.018608      168.8   
1     0.017084       171.2   0.017990   0.018405    0.018218      169.3   
2     0.017452       172.2   0.017352   0.018516    0.017827      170.0   
3     0.017820       172.7   0.016715   0.018627    0.017437      171.0   
4     0.017477       172.7   0.017084   0.017990    0.017723      170.9   

   cpi_lag_6  cpi_lag_12  
0      168.1       164.7  
1      168.4       164.7  
2      168.8       164.8  
3      169.3  

In [ ]:
from keras.models import Sequential
from keras.layers import Dense, Flatten, Conv1D, MaxPooling1D
import numpy as np

features = [
    "tariff_rate",
    "tau_lag_3","tau_lag_6","tau_lag_12",
    "cpi_lag_3","cpi_lag_6","cpi_lag_12"
]

target = "cpi_future"

data = century_data.dropna(subset=features + [target])

X = data[features].values
y = data[target].values

X = X.reshape((X.shape[0], 1, X.shape[1]))

n_steps = X.shape[1]
n_features = X.shape[2]

model = Sequential()
model.add(Conv1D(filters=64, kernel_size=1, activation='relu',
                 input_shape=(n_steps, n_features)))
model.add(MaxPooling1D(pool_size=1))
model.add(Flatten())
model.add(Dense(50, activation='relu'))
model.add(Dense(1))

model.compile(optimizer='adam', loss='mse')

model.fit(X, y, epochs=100, verbose=1)

x_input = X[0].reshape((1, n_steps, n_features))
yhat = model.predict(x_input)
print(yhat)

c:\Users\owass\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 82615.3906 
Epoch 2/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 24378.4375
Epoch 3/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 2385.6592
Epoch 4/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 586.4530
Epoch 5/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 792.8497  
Epoch 6/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 64.7853  
Epoch 7/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 87.5768 
Epoch 8/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 36.2652 
Epoch 9/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 17.2132
Epoch 10/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 16.5334 
Epoch 11/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 12.6141
Epoch 12/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 12.7878 
Epoch 13/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 12.3560 
Epoch 14/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 12.3835
Epoch 15/100
10/10 ━━━━━━━━━━━━━━━━

In [9]:
print("Predicted:", yhat)
print("Actual:", y[0])

Predicted: [[170.27335]]
Actual: 170.9
